# Network Intrusion Detection: v4 (two-stage model)

**What the real data showed**
- Train is 94.5% Availability, and almost all of it (79,698 rows) is one signature on port 21: a SYN flood with one 120-byte forward packet and fixed headers/window size.
- Train and test are almost perfectly separable (adversarial AUC ~ 1.0). The shift is in **timing / rate features** (duration, IAT, active/idle, `*_per_sec`) and **response-side features** (`bwd_*`, RST/ACK counts).
- On port 21, test rows have the **same forward-packet signature** as train but different timing/response behavior.
- But timing/rate features are *needed* to tell Benign / Reconnaissance / Spoofing apart (grouped-CV macro-F1 drops from ~0.72 to ~0.52 without them).

**So v4 uses each feature family where it is trustworthy**
1. **Stage 1: Availability vs not.** Uses only robust forward-packet features (no timing, rate, response, or ports).
2. **Stage 2: Benign vs Reconnaissance vs Spoofing** on the remaining rows, with the full feature set (timing included), because that is where it helps.
3. Feature sets for each stage are chosen by cluster-grouped CV **and** a label-free shift measure (adversarial AUC).
4. Saves `submission_v4.csv` plus alternates that change only the stage-2 feature set, so the leaderboard can arbitrate.

Runtime: about 4-6 minutes.

In [ ]:
import os, glob, warnings, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

SEED = 42
TARGET = "Target"
ID = "id"
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
T0 = time.time()

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"./{name}") + glob.glob(f"./data/{name}") + glob.glob(f"../data/{name}") + glob.glob(f"../../data/{name}")
    if not hits:
        raise FileNotFoundError(f"{name} not found. Add the competition data to the notebook.")
    return hits[0]

train = pd.read_csv(find("train.csv"))
test = pd.read_csv(find("test.csv"))
sample = pd.read_csv(find("sample_submission.csv"))
print("train:", train.shape, "| test:", test.shape, "| sample:", sample.shape)

In [ ]:
classes = sorted(train[TARGET].unique())            # Availability, Benign, Reconnaissance, Spoofing
cls2i = {c: i for i, c in enumerate(classes)}
K = len(classes)
y = train[TARGET].map(cls2i).values
train_mix = train[TARGET].value_counts(normalize=True).reindex(classes)
print(cls2i)
print(train[TARGET].value_counts())

AV = cls2i["Availability"]
is_av = (y == AV)
rest_classes = [c for c in classes if c != "Availability"]
K3 = len(rest_classes)
y3 = np.full(len(y), -1)
for j, c in enumerate(rest_classes):
    y3[y == cls2i[c]] = j
nonav_idx = np.where(~is_av)[0]

def prepare(train, test, drop=()):
    """Same feature matrix for train and test; categoricals share one category set; inf -> NaN."""
    feats = [c for c in train.columns if c not in (ID, TARGET) and c not in drop]
    full = pd.concat([train[feats], test[feats]], ignore_index=True)
    full = full.replace([np.inf, -np.inf], np.nan)
    for c in full.select_dtypes(exclude="number").columns:
        full[c] = full[c].astype("category")
    return full.iloc[:len(train)].reset_index(drop=True), full.iloc[len(train):].reset_index(drop=True)

X_tr, X_te = prepare(train, test)
PORTS = [c for c in ["id.orig_p", "id.resp_p"] if c in X_tr.columns]
num_cols = X_tr.select_dtypes("number").columns.tolist()
beh_cols = [c for c in num_cols if c not in PORTS]
print("features:", X_tr.shape[1], "| ports:", PORTS)

## 1. Pseudo-groups and shift-aware folds

Within each class, cluster rows on behavior features (all numeric except ports; log-scaled, standardized, PCA). Each cluster stands in for a "behavior profile", and `StratifiedGroupKFold` keeps each cluster inside one fold.

In [ ]:
Z = X_tr[beh_cols].copy()
Z = Z.fillna(Z.median())
Z = np.sign(Z) * np.log1p(Z.abs())
Z = StandardScaler().fit_transform(Z)
Z = PCA(n_components=min(20, Z.shape[1]), random_state=SEED).fit_transform(Z)

groups = np.zeros(len(y), dtype=int)
offset = 0
for ci, cls in enumerate(classes):
    idx = np.where(y == ci)[0]
    k = int(np.clip(len(idx) // 1500, 6, 20))
    km = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3, batch_size=4096).fit(Z[idx])
    groups[idx] = km.labels_ + offset
    offset += k

folds = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED).split(X_tr, y, groups))
for i, (tr, va) in enumerate(folds):
    print(f"fold {i}: val class counts {np.bincount(y[va], minlength=K)} | classes in train part: {len(np.unique(y[tr]))}/{K}")

## 2. Feature sets and helpers

- **Time / rate:** duration, `*_pkts_per_sec`, `payload_bytes_per_second`, `*_iat.*`, `active.*`, `idle.*`, bulk rates.
- **Response-side:** `bwd_*`, `down_up_ratio`, RST and ACK flag counts.

In [ ]:
A = list(X_tr.columns)
B = [c for c in A if c not in PORTS]
TIME_RATE = [c for c in B if ("per_sec" in c or "per_second" in c or "_iat." in c or c.startswith(("active.", "idle."))
                              or c == "flow_duration" or "bulk_rate" in c)]
RESPONSE = [c for c in B if c.startswith("bwd_") or c in ("down_up_ratio", "flow_RST_flag_count", "flow_ACK_flag_count")]
D = [c for c in B if c not in TIME_RATE]
E = [c for c in D if c not in RESPONSE]
F = D + PORTS
feature_sets = {"A": A, "B": B, "D": D, "E": E, "F": F}
desc = {"A": "all features", "B": "no ports", "D": "no ports, no timing/rate",
        "E": "no ports, no timing/rate/response", "F": "ports, no timing/rate"}
for k, f in feature_sets.items():
    print(f"  {k}: {desc[k]:40s} {len(f):3d} features")

lgb_params = dict(n_estimators=250, learning_rate=0.05, num_leaves=15, min_child_samples=100,
                  subsample=0.8, subsample_freq=1, colsample_bytree=0.6, reg_lambda=5.0,
                  random_state=SEED, n_jobs=-1, verbose=-1)
# Stage 2 has few rows (about 4.6k), so use smaller leaves
lgb2_params = dict(lgb_params, n_estimators=300, min_child_samples=20)

def sample_w(y_tr, power, k):
    """power 0 = none, 0.5 = sqrt-balanced, 1 = fully balanced."""
    cnt = np.bincount(y_tr, minlength=k).astype(float)
    cnt[cnt == 0] = 1.0
    w = (len(y_tr) / (k * cnt)) ** power
    return w[y_tr]

def adv_auc(Xa, Xb, feats, n_est=100):
    """Label-free shift measure: can a model tell rows of Xa from rows of Xb using these features?"""
    X = pd.concat([Xa[feats], Xb[feats]], ignore_index=True)
    yy = np.r_[np.zeros(len(Xa)), np.ones(len(Xb))]
    oof = np.zeros(len(yy))
    for tr, va in StratifiedKFold(3, shuffle=True, random_state=SEED).split(X, yy):
        m = lgb.LGBMClassifier(n_estimators=n_est, learning_rate=0.1, num_leaves=31, min_child_samples=50,
                               subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                               random_state=SEED, n_jobs=-1, verbose=-1).fit(X.iloc[tr], yy[tr])
        oof[va] = m.predict_proba(X.iloc[va])[:, 1]
    return roc_auc_score(yy, oof)

## 3. Stage 1: Availability vs everything else

Scored with the cluster-grouped CV. Pick: among sets within 0.005 of the best Availability F1, take the least shifted (lowest adversarial AUC between train and test).

In [ ]:
def run_stage1(feats):
    oof = np.zeros(len(y))
    for tr, va in folds:
        m = lgb.LGBMClassifier(**lgb_params).fit(X_tr.iloc[tr][feats], is_av[tr].astype(int))
        oof[va] = m.predict_proba(X_tr.iloc[va][feats])[:, 1]
    return oof

s1_oof, s1_rows = {}, []
for k, feats in feature_sets.items():
    s1_oof[k] = run_stage1(feats)
    pred = s1_oof[k] >= 0.5
    s1_rows.append(dict(set=k, desc=desc[k], n_features=len(feats),
                        avail_f1=f1_score(is_av, pred), precision=precision_score(is_av, pred),
                        recall=recall_score(is_av, pred), adv_auc=adv_auc(X_tr, X_te, feats)))
    print(f"set {k}: Availability F1 {s1_rows[-1]['avail_f1']:.4f} | adversarial AUC {s1_rows[-1]['adv_auc']:.4f}  [{time.time()-T0:.0f}s]")

s1 = pd.DataFrame(s1_rows).set_index("set").round(4)
best_f1 = s1["avail_f1"].max()
s1["eligible"] = s1["avail_f1"] >= best_f1 - 0.005
print()
print(s1)
s1_set = s1[s1["eligible"]]["adv_auc"].idxmin()
s1_feats = feature_sets[s1_set]
print("\nStage-1 feature set:", s1_set, "-", desc[s1_set])

# Fit stage 1 on all of train, score the test set
m1 = lgb.LGBMClassifier(**lgb_params).fit(X_tr[s1_feats], is_av.astype(int))
p_av_test = m1.predict_proba(X_te[s1_feats])[:, 1]
test_nonav = p_av_test < 0.5
print(f"Test rows flagged Availability by stage 1: {(~test_nonav).sum()} ({(~test_nonav).mean():.1%})")

## 4. Stage 2: Benign vs Reconnaissance vs Spoofing

Trained only on non-Availability rows. Scored on true non-Availability rows with the grouped CV (3-class macro-F1).
The shift measure here compares train non-Availability rows against the test rows stage 1 did *not* flag as Availability.

In [ ]:
def run_stage2(feats, power):
    oof = np.zeros((len(y), K3))
    for tr, va in folds:
        trn = tr[~is_av[tr]]
        m = lgb.LGBMClassifier(**lgb2_params).fit(X_tr.iloc[trn][feats], y3[trn], sample_weight=sample_w(y3[trn], power, K3))
        oof[np.ix_(va, m.classes_)] = m.predict_proba(X_tr.iloc[va][feats])
    return oof

def macro3(P):
    return f1_score(y3[nonav_idx], P[nonav_idx].argmax(1), average="macro")

Xn_tr = X_tr.iloc[nonav_idx].reset_index(drop=True)
Xn_te = X_te[test_nonav].reset_index(drop=True)

s2_oof, s2_rows = {}, []
for k, feats in feature_sets.items():
    s2_oof[k] = run_stage2(feats, 0.5)
    pc = f1_score(y3[nonav_idx], s2_oof[k][nonav_idx].argmax(1), average=None)
    s2_rows.append(dict(set=k, desc=desc[k], n_features=len(feats), macro_f1_3class=macro3(s2_oof[k]),
                        **{c: v for c, v in zip(rest_classes, pc)}, adv_auc=adv_auc(Xn_tr, Xn_te, feats)))
    print(f"set {k}: 3-class macro-F1 {s2_rows[-1]['macro_f1_3class']:.4f} | adversarial AUC (non-Avail domain) {s2_rows[-1]['adv_auc']:.4f}  [{time.time()-T0:.0f}s]")

s2 = pd.DataFrame(s2_rows).set_index("set").round(4)
best_s2 = s2["macro_f1_3class"].max()
s2["eligible"] = s2["macro_f1_3class"] >= best_s2 - 0.02
print()
print(s2)
s2_set = s2[s2["eligible"]]["adv_auc"].idxmin()
s2_feats = feature_sets[s2_set]
print("\nStage-2 feature set:", s2_set, "-", desc[s2_set])

In [ ]:
p2 = {0.5: s2_oof[s2_set]}
for power in (0.0, 1.0):
    p2[power] = run_stage2(s2_feats, power)
for power in sorted(p2):
    print(f"class-weight power {power}: 3-class macro-F1 {macro3(p2[power]):.4f}")
s2_power = max(p2, key=lambda p: macro3(p2[p]))
P2 = p2[s2_power]
print("Best power:", s2_power)

# Optional class-prior tuning on the 3 minority classes (kept only if it clearly helps)
def tune_weights(P, grid=np.exp(np.linspace(np.log(0.25), np.log(4), 17)), rounds=3):
    w = np.ones(K3)
    best = macro3(P)
    for _ in range(rounds):
        for k in range(K3):
            for g in grid:
                w2 = w.copy()
                w2[k] = g
                s = macro3(P * w2)
                if s > best + 1e-6:
                    best, w = s, w2
    return w, best

w_full, s_full = tune_weights(P2)
w_half = w_full ** 0.5
s_none, s_half = macro3(P2), macro3(P2 * w_half)
print(f"class-prior weights: none {s_none:.4f} | half-strength {s_half:.4f} | full {s_full:.4f}")
if s_half - s_none >= 0.005:
    w2_final, w2_name = w_half, "half-strength tuned"
else:
    w2_final, w2_name = np.ones(K3), "none"
print("Using:", w2_name, {c: round(float(v), 2) for c, v in zip(rest_classes, w2_final)})

## 5. Combined out-of-fold result

In [ ]:
p1_oof = s1_oof[s1_set]
rest_idx = np.array([cls2i[c] for c in rest_classes])
pred4 = np.where(p1_oof >= 0.5, AV, rest_idx[(P2 * w2_final).argmax(1)])
print(f"Two-stage grouped-CV macro-F1 (4 classes): {f1_score(y, pred4, average='macro'):.4f}")
print(classification_report(y, pred4, target_names=classes, digits=3))
print(pd.DataFrame(confusion_matrix(y, pred4), index=classes, columns=classes))

# Reference: single-stage LightGBM, no ports, all other features (what v1/v3 style models do)
ref = np.zeros((len(y), K))
for tr, va in folds:
    cnt = np.bincount(y[tr], minlength=K).astype(float)
    w = ((len(tr) / (K * cnt)) ** 0.5)[y[tr]]
    m = lgb.LGBMClassifier(**lgb_params).fit(X_tr.iloc[tr][B], y[tr], sample_weight=w)
    ref[np.ix_(va, m.classes_)] = m.predict_proba(X_tr.iloc[va][B])
print(f"\nReference single-stage (set B) grouped-CV macro-F1: {f1_score(y, ref.argmax(1), average='macro'):.4f}")
print(f"[{time.time()-T0:.0f}s elapsed]")

## 6. Final fit and submission (`submission_v4.csv`)

In [ ]:
def fit_stage2_predict(feats, power, weights):
    m = lgb.LGBMClassifier(**lgb2_params).fit(X_tr.iloc[nonav_idx][feats], y3[nonav_idx],
                                              sample_weight=sample_w(y3[nonav_idx], power, K3))
    return m.predict_proba(X_te[feats]) * weights

def make_labels(P3):
    out = np.where(p_av_test >= 0.5, AV, rest_idx[P3.argmax(1)])
    return np.array(classes)[out]

P3_test = fit_stage2_predict(s2_feats, s2_power, w2_final)
labels = make_labels(P3_test)

sub = pd.DataFrame({ID: test[ID].values, TARGET: labels})
path = os.path.join(OUT, "submission_v4.csv")
sub.to_csv(path, index=False)
assert list(sub.columns) == list(sample.columns) == [ID, TARGET]
assert len(sub) == len(sample) and set(sub[ID]) == set(sample[ID])
assert sub[TARGET].isin(classes).all()
print("Saved", path)

pd.DataFrame(P3_test, columns=rest_classes).assign(p_availability=p_av_test, **{ID: test[ID].values}).to_csv(
    os.path.join(OUT, "test_probs_v4.csv"), index=False)

v4_mix = sub[TARGET].value_counts(normalize=True).reindex(classes).fillna(0)
print(pd.DataFrame({"train": train_mix, "test_pred_v1": pd.Series({"Availability": 0.029, "Benign": 0.599, "Reconnaissance": 0.225, "Spoofing": 0.148}).reindex(classes),
                    "test_pred_v4": v4_mix}).round(3))

port_view = [21.0, 53.0, 80.0, 443.0, 1883.0, 5353.0]
mask = test["id.resp_p"].isin(port_view).values
print("\nsubmission_v4 predicted class by test port (row shares):")
print(pd.crosstab(test.loc[mask, "id.resp_p"].astype(int), sub.loc[mask, TARGET], normalize="index").round(2))
print("\nShare of test rows where the stage-2 top probability < 0.6 (non-Availability rows only):",
      round(float((P3_test[test_nonav].max(1) / P3_test[test_nonav].sum(1) < 0.6).mean()), 3))

## 7. Alternates (change only the stage-2 feature set)

Each file keeps the same stage 1 and swaps the stage-2 feature set. Comparing their public scores shows how much timing/rate features help (or hurt) on the hidden profiles.

In [ ]:
alt_rows = {}
for k in ["A", "B", "D", "F"]:
    if k == s2_set:
        continue
    lab = make_labels(fit_stage2_predict(feature_sets[k], s2_power, np.ones(K3)))
    pd.DataFrame({ID: test[ID].values, TARGET: lab}).to_csv(os.path.join(OUT, f"submission_v4_stage2_{k}.csv"), index=False)
    alt_rows[f"stage2={k}"] = pd.Series(lab).value_counts(normalize=True).reindex(classes).fillna(0)
alt_rows[f"v4 pick (stage2={s2_set})"] = v4_mix
print(pd.DataFrame(alt_rows).round(3))

pick = sub[TARGET].values
for k in ["A", "B", "D", "F"]:
    f = os.path.join(OUT, f"submission_v4_stage2_{k}.csv")
    if os.path.exists(f):
        other = pd.read_csv(f)[TARGET].values
        print(f"stage2={k}: agrees with v4 pick on {np.mean(other == pick):.1%} of test rows")
print(f"\nTotal time: {time.time()-T0:.0f}s")

## What to record and how to read it

- The stage-1 and stage-2 tables (CV score and adversarial AUC per feature set).
- The two-stage combined CV vs the single-stage reference.
- The predicted class mix and the share of port-21 test rows predicted Availability.
- Public leaderboard scores for `submission_v4.csv` and the alternates.

**Caveat:** the grouped CV can't fully reproduce the timing shift, and the public and private leaderboards use *different* hidden profiles. Treat the public score as one signal, and prefer the pick that is robust in principle.